In [ ]:
# Importar SparkSession
from pyspark.sql import SparkSession

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("ActividadStreaming") \
    .config(
        # Descarga automática del conector Spark-Kafka
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"
    ) \
    .getOrCreate()

# Suprimir logs informativos, mostrar solo errores
spark.sparkContext.setLogLevel("ERROR")

# Confirmar que Spark está listo e imprimir la versión instalada
print("Spark listo:", spark.version)

In [ ]:
# Crear un DataFrame de streaming que lee mensajes en tiempo real desde Kafka
# format kafka indica que la fuente de datos es Kafka
# kafka.bootstrap.servers es la dirección del broker dentro de la red Docker
# subscribe indica el topic del que se van a leer los mensajes
# startingOffsets earliest lee todos los mensajes desde el inicio del topic
df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "actividad-topic") \
    .option("startingOffsets", "earliest") \
    .load()

# Los mensajes de Kafka llegan en bytes
# CAST(value AS STRING) los convierte a texto legible
# AS mensaje renombra la columna para mayor claridad
df_mensajes = df_stream.selectExpr("CAST(value AS STRING) AS mensaje")

In [ ]:
# Iniciar la escritura continua del stream en memoria
# outputMode append agrega cada nuevo mensaje sin borrar los anteriores
# format memory guarda los datos en RAM para consultarlos con SQL
# queryName es el nombre de la tabla para consultarla con spark.sql()
query = df_mensajes.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("actividad_table") \
    .start()

In [ ]:
import time
from IPython.display import clear_output

# Ejecutar 10 consultas con 3 segundos de pausa entre cada una
for i in range(10):
    # Limpia la pantalla antes de mostrar la nueva consulta
    clear_output(wait=True)
    print(f"Consulta #{i+1} — mensajes recibidos:")

    # Consultar la tabla en memoria donde Spark almacena los mensajes
    # truncate=False muestra el texto completo sin cortar
    spark.sql("SELECT * FROM actividad_table").show(truncate=False)

    # Esperar 3 segundos antes de la siguiente consulta
    time.sleep(3)